# Chapitre 16 - Cours 2 : apprendre un ensemble, puis le post-traiter

Ce second notebook couvre la section 16.3 du livre. L'idée, due à Friedman et
Popescu, est de **séparer** les deux tâches de l'apprentissage ensembliste :

1. construire un dictionnaire **fini** de fonctions de base (des arbres) à partir des
   données ;
2. les **combiner** en ajustant un chemin de lasso dans ce dictionnaire.

Nous verrons le post-traitement d'une forêt aléatoire sur les données spam
(figure 16.6), l'algorithme ISLE de génération d'ensembles (algorithme 16.2,
figure 16.8) et les **ensembles de règles** (RuleFit simplifié, figure 16.10).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch16_ensembles import (standardize, lasso_path_cd, lasso_path_fast,
                                isle_ensemble, tree_predictions,
                                staged_ensemble_predictions, extract_rules,
                                rule_matrix, rule_to_str)

## 1. Le modèle et le post-traitement lasso

On considère des fonctions de la forme (éq. (16.8) du livre)
$$
f(x)=\alpha_0+\sum_{T_k\in\mathcal T}\alpha_k\,T_k(x),
$$
où $\mathcal T$ est un dictionnaire de fonctions de base, typiquement des arbres. Le
cours 1 a montré que le gradient boosting avec rétrécissement suit un chemin
régularisé $L_1$ monotone dans l'espace de **tous** les arbres. Friedman et Popescu
proposent une approche hybride en deux étapes :

- induire à partir des données un dictionnaire fini
  $\mathcal T_L=\{T_1(x),\dots,T_M(x)\}$ ;
- ajuster un chemin de lasso dans ce dictionnaire (éq. (16.9)) :
$$
\alpha(\lambda)=\arg\min_\alpha\ \sum_{i=1}^N L\Big[y_i,\ \alpha_0+\sum_{m=1}^M\alpha_mT_m(x_i)\Big]+\lambda\sum_{m=1}^M|\alpha_m|.
$$